# Check Dataset Split

## Example config

In [ ]:
# Example

URIS = {
    "karolinska_calibration": "mlflow-artifacts:/38/53aa1150ec7a4df8a90351f54585ba99/artifacts/PANDA - Karolinska_calibration.csv",
    "karolinska_test": "mlflow-artifacts:/38/53aa1150ec7a4df8a90351f54585ba99/artifacts/PANDA - Karolinska_test.csv",
    "karolinska_total": "mlflow-artifacts:/38/9fdcbc70346646f9823b6cf72ec84e7e/artifacts/karolinska_metadata.csv",
}

CHECKS = {
    ("karolinska_calibration", "karolinska_test"): lambda df1, df2: (
        set(df1["slide_id"]) & set(df2["slide_id"]) == set()
    ),
    (
        "karolinska_calibration",
        "karolinska_test",
        "karolinska_total",
    ): lambda df1, df2, df3: len(df1) + len(df2) == len(df3),
}

## Setup

In [5]:
import os

import mlflow
import pandas as pd


IS_PUBLIC = True  # TBD
URIS = {}  # TBD

CHECKS = {}  # TBD

if IS_PUBLIC:
    os.environ["MLFLOW_TRACKING_USERNAME"] = "524839@mail.muni.cz"  # TBD
    os.environ["MLFLOW_TRACKING_PASSWORD"] = "VG9PLfHXXHOMsQnXtbq3UOF7"  # TBD
    mlflow.set_tracking_uri("https://mlflow.rationai.cloud.e-infra.cz/")

dfs = {
    k: pd.read_csv(mlflow.artifacts.download_artifacts(uri)) for k, uri in URIS.items()
}

In [6]:
def carcinoma_counts(metadata: pd.DataFrame) -> tuple[int, int]:
    return (len(metadata[metadata["carcinoma"]])), (
        len(metadata[not metadata["carcinoma"]])
    )

In [ ]:
for name, df in dfs.items():
    print(name, len(df), carcinoma_counts(df))

In [8]:
for check_names, check in CHECKS.items():
    assert check(*[dfs[name] for name in check_names])

In [8]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def plot_gs_distr(dfs: dict[str, pd.DataFrame]) -> None:

    def gleason_sort_key(s: str) -> tuple[int, int]:
        if s.lower() == "negative":
            return (-1, 0)  # always first
        parts = s.split("+")
        return (int(parts[0]), int(parts[1]))

    # Compute normalized value counts for each df
    counts_per_df = {
        name: df["gleason_score"].value_counts(normalize=True)
        for name, df in dfs.items()
    }

    # Union of all categories across all dfs, sorted consistently
    all_categories = set()
    for counts in counts_per_df.values():
        all_categories.update(counts.index)
    sorted_categories = sorted(all_categories, key=gleason_sort_key)

    # Align each df's counts to the shared category order (missing -> 0)
    aligned = {
        name: counts.reindex(sorted_categories, fill_value=0)
        for name, counts in counts_per_df.items()
    }

    # Plot grouped bars
    n_groups = len(sorted_categories)
    n_series = len(dfs)
    x = np.arange(n_groups)
    total_width = 0.8
    bar_width = total_width / n_series

    plt.figure(figsize=(12, 6))
    for i, (name, values) in enumerate(aligned.items()):
        offset = (i - (n_series - 1) / 2) * bar_width
        plt.bar(x + offset, values.values, width=bar_width, label=name)

    plt.xlabel("Gleason Score")
    plt.ylabel("Proportion")
    plt.title("Distribution of Gleason Scores")
    plt.xticks(x, sorted_categories, rotation=45, ha="right")
    plt.legend(title="Dataset")
    plt.tight_layout()
    plt.show()

In [ ]:
plot_gs_distr(dfs)

If Gleason score distributions match, carcinoma must match as well (we just add the non-negative cases).

In [10]:
import pandas as pd


def plot_binary_distr(dfs: dict[str, pd.DataFrame]) -> None:
    # Compute normalized value counts for each df
    counts_per_df = {
        name: df["carcinoma"].value_counts(normalize=True) for name, df in dfs.items()
    }

    # Union of all categories across all dfs, sorted consistently (e.g. False, True)
    all_categories = set()
    for counts in counts_per_df.values():
        all_categories.update(counts.index)
    sorted_categories = sorted(all_categories, key=str)

    # Align each df's counts to the shared category order (missing -> 0)
    aligned = {
        name: counts.reindex(sorted_categories, fill_value=0)
        for name, counts in counts_per_df.items()
    }

    # Plot grouped bars
    n_groups = len(sorted_categories)
    n_series = len(dfs)
    x = np.arange(n_groups)
    total_width = 0.6
    bar_width = total_width / n_series

    plt.figure(figsize=(8, 6))
    for i, (name, values) in enumerate(aligned.items()):
        offset = (i - (n_series - 1) / 2) * bar_width
        plt.bar(x + offset, values.values, width=bar_width, label=name)

    plt.xlabel("Carcinoma")
    plt.ylabel("Fraction")
    plt.title("Carcinoma distribution")
    plt.xticks(x, [str(c) for c in sorted_categories])
    plt.legend(title="Dataset")
    plt.tight_layout()
    plt.show()

In [ ]:
plot_binary_distr(dfs)